### Data Enrichment

In [0]:
%pip install faker

**Note**: Since I dont have drivers_id column for drivers table, I decided to generate car numbers instead and join based on them.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from faker import Faker

df_taxi = spark.read.table("unified_lakehouse.silver.taxi")
df_drivers = spark.read.table("unified_lakehouse.silver.drivers")

fake = Faker()

@F.udf(StringType())
def generate_car_number(_):
    return fake.license_plate()

df_taxi_synthetic = df_taxi.withColumn("car_number", generate_car_number(F.lit(1)))

gold_enriched_df = (
    df_taxi_synthetic
    .join(df_drivers.drop("ingest_time"), on="car_number", how="left")
)

gold_enriched_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("unified_lakehouse.gold.enriched_trips")